In [20]:
import yfinance as yf
import pandas as pd
from datetime import datetime, timedelta
import numpy as np

start_date = datetime.now() - timedelta(days = 5*365)
df = yf.download("BTC-USD", start=start_date, end=datetime.now(), interval = "1d", auto_adjust=True)

if not df.empty:
    df.to_csv("BTC-USD_5_years")
    df.head()
else:
    print("No data found")

[*********************100%***********************]  1 of 1 completed


In [21]:
df.columns = [c[0] if isinstance(c, tuple) else c for c in df.columns]
df['Daily Return'] = df['Close'].pct_change() #Daily Return
df['MA_7'] = df['Close'].rolling(window=7).mean() #MA 7 days period
df['MA_30'] = df['Close'].rolling(window=30).mean() #MA 30 days period
df['Volatility_30'] = df['Daily Return'].rolling(window=30).std() #Moving volatility 30d period

#

In [26]:

def compute_rsi(close: 'pd.Series', length: int = 14) -> 'pd.Series':
    """
    Compute the Relative Strength Index (RSI) using Wilder's smoothing.
    Equivalent to pandas_ta.rsi(length=length).
    """
    delta = close.diff()
    gain = delta.clip(lower=0)
    loss = -delta.clip(upper=0)

    # Wilder's smoothing == EMA with alpha = 1/length
    avg_gain = gain.ewm(alpha=1 / length, adjust=False, min_periods=length).mean()
    avg_loss = loss.ewm(alpha=1 / length, adjust=False, min_periods=length).mean()

    rs = avg_gain / avg_loss
    rsi = 100 - (100 / (1 + rs))
    return rsi

def compute_macd(close: 'pd.Series', fast: int = 12, slow: int = 26, signal: int = 9) -> 'pd.DataFrame':
    """
    Compute MACD (Moving Average Convergence Divergence).
    Returns a DataFrame with columns: MACD, Signal, Histogram
    Equivalent to pandas_ta.macd(fast=fast, slow=slow, signal=signal).
    """
    ema_fast = close.ewm(span=fast, adjust=False).mean()
    ema_slow = close.ewm(span=slow, adjust=False).mean()

    macd_line = ema_fast - ema_slow
    signal_line = macd_line.ewm(span=signal, adjust=False).mean()
    histogram = macd_line - signal_line

    # Return as DataFrame matching pandas_ta output column names
    result = pd.DataFrame({
        'MACD_12_26_9': macd_line,
        'MACDh_12_26_9': histogram,
        'MACDs_12_26_9': signal_line,
    })
    return result

df['RSI'] = compute_rsi(df['Close'])
macd_df = compute_macd(df['Close'], fast=12, slow=26, signal=9)
df = df.join(macd_df)

In [36]:
df['target_up_tomorrow'] = (df['Daily Return'].shift(-1) > 0)
df.dropna()

,Close,High,Low,Open,Volume,Daily Return,MA_7,MA_30,Volatility_30,RSI,MACD_12_26_9,MACDh_12_26_9,MACDs_12_26_9,target_up_tomorrow
Date,,,,,,,,,,,,,,
2021-08-03,38152.980469,39750.031250,37782.050781,39178.402344,26189830450,-0.026758,40170.841518,34923.283854,0.032122,48.271523,1716.573530,434.915891,1281.657639,True
2021-08-04,39747.503906,39952.296875,37589.164062,38213.332031,25372562724,0.041793,40135.355469,35123.333854,0.031616,53.576499,1708.768939,341.689040,1367.079899,True
2021-08-05,40869.554688,41341.933594,37458.003906,39744.515625,35185031017,0.028229,40258.374442,35344.479167,0.031843,56.924248,1772.689405,324.487604,1448.201800,True
2021-08-06,42816.500000,43271.660156,39932.179688,40865.867188,38226483046,0.047638,40341.367746,35643.184896,0.032528,62.039561,1957.879830,407.742424,1550.137406,True
2021-08-07,44555.800781,44689.859375,42618.566406,42832.796875,40030862141,0.040622,40759.882812,36032.465885,0.032258,65.931757,2219.407663,535.416206,1683.991458,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2026-06-28,59532.339844,60432.707031,58879.632812,59940.351562,16282230874,-0.006803,60975.114397,63955.734766,0.022450,30.696435,-2355.874666,-47.913098,-2307.961568,True
2026-06-29,60138.378906,60682.339844,58856.187500,59522.789062,30829983083,0.010180,60430.296317,63501.852865,0.022559,33.987995,-2306.378633,1.266348,-2307.644981,False
2026-06-30,58558.859375,60173.218750,58111.671875,60136.453125,32747419391,-0.026265,59843.273438,63001.158594,0.022827,29.990085,-2367.317731,-47.738200,-2319.579531,True
